In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [3]:
!wget $data

--2026-10-13 00:09:41--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.5’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.1s    

2026-10-13 00:09:42 (2.06 MB/s) - ‘course_lead_scoring_2026.csv.5’ saved [278746/278746]



In [4]:
df = pd.read_csv(data)

In [5]:
df.dtypes



lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
categorical_features = ['lead_source','industry','employment_status','location']
numerical_features = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
df[categorical_features].head(5) 

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NaN
4,referral,manufacturing,student,north_america


In [7]:
df[numerical_features].head(5)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,88160.0,3,4,0.64
1,72688.0,3,7,0.72
2,44697.0,3,4,0.58
3,NaN,3,6,0.57
4,24062.0,4,5,0.62


In [8]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [9]:
### Fill missing values of categorical feature by NA category 
df[categorical_features] = df[categorical_features].fillna('NA')

In [10]:
### Fill missing values of numerical feature by 0 
df[numerical_features] = df[numerical_features].fillna(0)

In [11]:
# Q1: What is the most frequent observation (mode) for the column industry?
df['industry'].value_counts().index[0]

'technology'

In [12]:
# q2: What are the two features that have the biggest correlation?
df[numerical_features].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [13]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
print(len(df_full_train), len(df_train), len(df_val))
len(df_full_train), len(df_test)

4000 3000 1000


(4000, 1000)

In [14]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values 

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [15]:
from sklearn.metrics import mutual_info_score
def mutual_info_score_table(series):
    return mutual_info_score(series, y_train)

In [16]:
df_train[categorical_features].apply(mutual_info_score_table)

lead_source          0.034346
industry             0.001995
employment_status    0.018966
location             0.001043
dtype: float64

In [17]:
#Q3: Which of these variables has the biggest mutual information score?
mutual_info_table = df_train[categorical_features].apply(lambda col: mutual_info_score(col, y_train))
mutual_info_table.round(2).sort_values(ascending=False).index[0]


'lead_source'

In [18]:
from sklearn.feature_extraction import DictVectorizer

In [19]:
train_dicts = df_train.to_dict(orient="records")

In [20]:
dv = DictVectorizer(sparse=False)

In [21]:
dv.fit(train_dicts)

DictVectorizer(sparse=False)

In [22]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [23]:
X_train  = dv.transform(train_dicts)
X_train

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       [4.0220e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [7.1532e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [4.8836e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]], shape=(3000, 28))

In [24]:
val_dicts = df_val.to_dict(orient="records")
X_val  = dv.transform(val_dicts)

In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [26]:
model.coef_, model.intercept_

(array([[-4.57849985e-06, -5.83434850e-04,  7.40346890e-03,
         -3.28471892e-03, -8.04069170e-03, -6.33360193e-03,
         -4.41226767e-04, -2.95759734e-03, -6.49404976e-04,
         -2.15453817e-03, -2.30751043e-03, -1.65985277e-03,
         -6.68848037e-04,  1.36728058e-01,  1.10036031e-02,
          9.51725098e-05, -8.62466509e-03,  6.94367905e-03,
         -1.37825294e-02,  8.71726672e-03, -4.18790228e-03,
          2.30545170e-04, -1.79050012e-03, -1.77472633e-03,
         -5.83764763e-04, -3.10519687e-03, -3.81533558e-03,
          5.37983416e-02]]),
 array([-0.01083898]))

In [49]:
y_pred = model.predict_proba(X_val)[:,1]

In [29]:
converted = (y_pred >=0.5)

In [33]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['pred'] =  converted.astype(int)
df_pred['real']= y_val
df_pred['correct'] = df_pred.pred == df_pred.real
df_pred

,probability,pred,real,correct
0,0.617257,1,1,True
1,0.443641,0,1,False
2,0.604340,1,0,False
3,0.645622,1,1,True
4,0.522142,1,0,False
...,...,...,...,...
995,0.684710,1,1,True
996,0.669580,1,1,True
997,0.458473,0,0,True
998,0.561770,1,0,False


In [46]:
#Q4:What accuracy did you get?
accuracy = df_pred.correct.mean()
round(float(accuracy), 2) 

0.65

In [60]:
accuracy

np.float64(0.645)

In [ ]:
# ----------------------------------------Question 5 -----------------------------------------------------
# Which of following feature has the smallest difference? 

In [55]:
def train_and_score(features):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    correct = y_pred == y_val
    accuracy = correct.mean()
    return accuracy

In [63]:
original_acc = train_and_score(categorical_features +  numerical_features)
print("original accuracy:", original_acc)

original accuracy: 0.645


In [74]:
total_features = categorical_features +  numerical_features
total_features
results = []
for features in total_features:
    subset = [x for x in total_features if x != features]
    acc = train_and_score(subset)
    results.append({
        "feature": features,
        "accuracy_without": acc,
        "difference": original_acc - acc,
    })
diff_df = pd.DataFrame(results).sort_values("difference")
print(diff_df)

                    feature  accuracy_without  difference
4             annual_income             0.724      -0.079
1                  industry             0.645       0.000
3                  location             0.645       0.000
2         employment_status             0.645       0.000
7                lead_score             0.644       0.001
5  number_of_courses_viewed             0.643       0.002
0               lead_source             0.642       0.003
6         interaction_count             0.601       0.044


In [ ]:
# # ----------------------------------------Question 6 -----------------------------------------------------
# Which of these C leads to the best accuracy on the validation set?

In [76]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return (y_pred == y_val).mean()

In [79]:
total_features = categorical_features + numerical_features

results = []
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(total_features, C=C)
    results.append({
        "C": C,
        "accuracy": round(float(acc), 3),
    })

c_df = pd.DataFrame(results).sort_values("accuracy")
c_df

,C,accuracy
0,0.000001,0.598
1,0.000010,0.598
2,0.000100,0.613
3,0.001000,0.645
